# Quick gNATSGO STAC Fetch Test

Tests `fetch_stac_cog()` with per-variable `asset_key` for gNATSGO rasters,
without running the full pipeline. Uses a small subset of DRB HRUs.

In [ ]:
import os

# Ensure PROJ_DATA is set for the Jupyter kernel (VS Code may not inherit pixi activation)
proj_data = os.path.join(os.getcwd(), ".pixi", "envs", "dev", "share", "proj")
if os.path.isdir(proj_data):
    os.environ["PROJ_DATA"] = proj_data

import geopandas as gpd

from hydro_param.data_access import fetch_stac_cog
from hydro_param.dataset_registry import DatasetEntry

In [ ]:
# Load a small subset of the DRB fabric for testing
fabric = gpd.read_file("../data/pywatershed_gis/drb_2yr/nhru.gpkg")
print(f"Full fabric: {len(fabric)} features, CRS={fabric.crs}")

# Take first 10 features for a quick test
subset = fabric.head(10)
bounds = subset.to_crs("EPSG:4326").total_bounds  # [minx, miny, maxx, maxy]
bbox = bounds.tolist()
print(f"Test bbox (WGS84): {bbox}")

In [ ]:
# gNATSGO dataset entry matching configs/datasets/soils.yml
entry = DatasetEntry(
    strategy="stac_cog",
    catalog_url="https://planetarycomputer.microsoft.com/api/stac/v1",
    collection="gnatsgo-rasters",
    crs="EPSG:5070",
    sign="planetary_computer",
)

# Test each variable with its per-variable asset_key
variables = {
    "aws0_100": "Available water storage 0-100cm",
    "rootznemc": "Root zone depth",
    "rootznaws": "Root zone available water storage",
}

In [ ]:
for var_name, description in variables.items():
    print(f"\n--- Fetching {var_name} ({description}) ---")
    da = fetch_stac_cog(entry, bbox, asset_key=var_name)
    print(f"  Shape: {da.shape}")
    print(f"  CRS: {da.rio.crs}")
    print(f"  Min: {float(da.min()):.4f}, Max: {float(da.max()):.4f}, Mean: {float(da.mean()):.4f}")
    print(f"  NoData count: {int(da.isnull().sum())} / {da.size}")

In [ ]:
# Verify the old default asset_key='data' would fail
try:
    fetch_stac_cog(entry, bbox)  # no asset_key override -> uses 'data'
    print("ERROR: Should have raised KeyError!")
except KeyError as e:
    print(f"Correctly raised KeyError: {e}")

In [ ]:
# Quick zonal stats test with one variable
import tempfile
from pathlib import Path

from hydro_param.data_access import save_to_geotiff
from hydro_param.processing import ZonalProcessor

da = fetch_stac_cog(entry, bbox, asset_key="aws0_100")

with tempfile.TemporaryDirectory() as tmpdir:
    tiff_path = Path(tmpdir) / "aws0_100.tif"
    save_to_geotiff(da, tiff_path)

    processor = ZonalProcessor()
    result = processor.process(
        fabric=subset,
        tiff_path=tiff_path,
        variable_name="aws0_100",
        id_field="nhm_id",
        engine="exactextract",
        statistics=["mean"],
        categorical=False,
        source_crs="EPSG:5070",
    )

print(f"\nZonal stats for aws0_100 ({len(result)} features):")
print(result.head(10))